# 06 — Evaluating the Supervised Model

## Objective
Load the SFT (Supervised Fine-Tuning) checkpoint, verify its original base-model and dataset identities, and compare its held-out answers with the untouched starting model. Preserve raw responses and diagnostic metrics before interpreting any performance difference.

An adapter is compatible only with the precise base checkpoint used during its training. Use a consistent source manifest, identical prompts, consistent inference settings, and matched validation IDs.

### Paths and source partition integrity

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


In [2]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter

DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def load_pubmedqa_source():
    """Load the labeled source, using the single existing Arrow cache in offline mode."""
    from datasets import Dataset, config, load_dataset
    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        cached_files = list(Path(config.HF_DATASETS_CACHE).glob("qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
        if len(cached_files) != 1:
            raise FileNotFoundError("Expected one cached PubMedQA labeled Arrow file; found " + str(len(cached_files)))
        print("Source mode: existing Hugging Face Arrow cache")
        return Dataset.from_file(str(cached_files[0]))
    print("Source mode: Hugging Face load_dataset")
    return load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

### Role-aware response format and exact-answer scoring

In [3]:
SYSTEM_INSTRUCTION = (
    "Answer the biomedical question using ONLY the supplied abstract evidence. "
    "Choose exactly one decision label: yes, no, or maybe. "
    "Return the label alone, without an explanation. "
    "Use maybe when the evidence does not establish yes or no."
)


def question_messages(row):
    """Use one consistent role-aware input protocol across model variants."""
    full_text = evidence_text(row)
    if not full_text:
        raise ValueError("Question has no evidence passage.")
    # Keep both the abstract introduction and conclusion; the conclusion often
    # carries the decision evidence. Apply the same limit in training/evaluation.
    if len(full_text) > 1600:
        text = full_text[:950] + "\n[Middle of abstract omitted for length]\n" + full_text[-600:]
    else:
        text = full_text
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"Question: {row['question']}\n\nAbstract evidence:\n{text}"},
    ]


def parse_decision(generated_text):
    """Parse an exact one-label answer; preserve raw outputs when parsing fails."""
    import re
    text = str(generated_text).strip()
    match = re.fullmatch(r"(?i)(yes|no|maybe)[.\s]*", text)
    return match.group(1).lower() if match else "invalid"

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


def assess_predictions(frame):
    """Report output validity, class balance, and independent majority baseline."""
    if frame.empty:
        raise ValueError("No predictions provided for scoring.")
    y_true = frame["label"].astype(str).str.lower()
    y_pred = frame["prediction"].astype(str).str.lower()
    if not set(y_true).issubset(set(LABELS)):
        raise ValueError("Reference labels must be yes/no/maybe.")
    counts = y_true.value_counts()
    majority_label = counts.idxmax()
    summary = {
        "n": len(frame),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(LABELS), average="macro", zero_division=0)),
        "valid_format": float(y_pred.isin(LABELS).mean()),
        "majority_label": majority_label,
        "majority_accuracy": float((y_true == majority_label).mean()),
        "uniform_random_accuracy": 1 / len(LABELS),
    }
    return summary


def show_diagnostics(frame):
    """Provide evidence to distinguish formatting errors from actual classification errors."""
    metrics = assess_predictions(frame)
    print("Sample size:", metrics["n"])
    print("Reference distribution:", frame["label"].value_counts().to_dict())
    print("Predicted distribution:", frame["prediction"].value_counts().to_dict())
    print("Summary:", metrics)
    print("Confusion matrix (rows=reference, columns=prediction):")
    labels = list(LABELS) + ["invalid"]
    print(pd.DataFrame(confusion_matrix(frame["label"], frame["prediction"], labels=labels),
                       index=labels, columns=labels))
    mistakes = frame.loc[frame["prediction"] != frame["label"]]
    inspect_columns = [c for c in ("id", "label", "prediction", "raw", "input_token_count") if c in mistakes]
    if not mistakes.empty:
        print("Error samples (inspect generation and label handling):")
        display(mistakes[inspect_columns].head(8))
    if metrics["n"] < 100:
        print("CAUTION: A sample of fewer than 100 records gives an imprecise performance estimate.")
    if metrics["accuracy"] <= metrics["majority_accuracy"]:
        print("CAUTION: The model does not beat the validation majority-class baseline.")
    return metrics

### Deterministic generation

In [5]:
# Use the tokenizer's instruction-chat format: raw plain-text prompts are not equivalent.
# Extract newly generated tokens only, and preserve the exact decoded text for auditing.
def generate_answer(model, tokenizer, row, max_input_tokens=768, max_output_tokens=8):
    import torch
    from transformers import GenerationConfig

    if tokenizer.chat_template is None:
        raise ValueError("Model tokenizer does not provide the expected chat template.")
    chat_prompt = tokenizer.apply_chat_template(
        question_messages(row), tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(
        chat_prompt,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=False,  # Chat template already contains special tokens.
    )
    input_length = int(encoded["input_ids"].shape[-1])
    if input_length > max_input_tokens:
        raise ValueError(
            f"Paper {paper_id(row)} exceeds the input token budget "
            f"({input_length} > {max_input_tokens}); revise the shared evidence budget. "
            "Do not truncate the assistant generation marker."
        )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    # Ignore sampling values embedded in a model repository's generation_config.json.
    # A freshly constructed config keeps this run deterministic and avoids warnings
    # about sampling-only flags when do_sample=False.
    config = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_output_tokens,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    with torch.inference_mode():
        sequence = model.generate(**encoded, generation_config=config)
    response = tokenizer.decode(sequence[0, input_length:], skip_special_tokens=True)
    return {"raw": response, "prediction": parse_decision(response), "input_token_count": input_length}

### Validate model lineage, run inference, and retain every answer

In [6]:
import os
import json

if os.getenv("RUN_MODEL_EVAL", "0") == "1":
    import gc
    import time
    import torch
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftConfig, PeftModel

    adapter_dir = RUNS / "sft_adapter"
    training_manifest = adapter_dir / "run_manifest.json"
    if not training_manifest.exists():
        raise FileNotFoundError("Missing trained SFT run manifest. Execute notebook 05 first.")
    metadata = json.loads(training_manifest.read_text(encoding="utf-8"))
    model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    if metadata["model_id"] != model_id:
        raise ValueError("SFT adapter base model differs from requested MODEL_ID.")
    adapter_config = PeftConfig.from_pretrained(str(adapter_dir))
    if adapter_config.base_model_name_or_path != model_id:
        raise ValueError("Saved adapter configuration names a different base model.")
    base_protocol = json.loads((RUNS / "predictions_base_validation_manifest.json").read_text(encoding="utf-8"))
    if metadata.get("model_revision") != base_protocol.get("model_revision"):
        raise ValueError("SFT training base revision differs from the evaluated starting checkpoint.")

    source = load_pubmedqa_source()
    partitions = prepare_partitions(source, create_manifest=False)
    fingerprint = json.loads(MANIFEST.read_text())["fingerprint"]
    if metadata["dataset_fingerprint"] != fingerprint:
        raise ValueError("SFT training data does not match the original-data manifest.")

    validation = sorted(partitions["validation"], key=paper_id)
    max_items = int(os.getenv("EVAL_MAX_ITEMS", "0"))
    if max_items > 0:
        validation = validation[:max_items]
        print("DIAGNOSTIC SUBSET: no full validation claim is supported.")

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    run_start = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
    if getattr(model.config, "_commit_hash", None) != metadata.get("model_revision"):
        raise ValueError("Loaded base model revision differs from SFT training lineage.")
    model = PeftModel.from_pretrained(model, str(adapter_dir)).eval()

    inference_start = time.perf_counter()
    results = []
    for index, row in enumerate(validation, 1):
        results.append({
            "id": paper_id(row),
            "label": str(row["final_decision"]).lower(),
            "question": str(row["question"]),
            **generate_answer(model, tokenizer, row),
            "model_id": model_id,
        })
        if index % 20 == 0 or index == len(validation):
            print(f"SFT examples completed: {index}/{len(validation)}")

    inference_seconds = time.perf_counter() - inference_start
    elapsed_seconds = time.perf_counter() - run_start
    peak_gpu_gib = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else None
    predictions = pd.DataFrame(results)
    show_diagnostics(predictions)
    predictions.to_csv(RUNS / "predictions_sft_validation.csv", index=False)
    protocol = {
        "model_variant": "sft",
        "model_id": model_id,
        "model_revision": metadata["model_revision"],
        "adapter_method": metadata["method"],
        "device": str(model.device),
        "model_dtype": str(model.dtype),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "seed": SEED,
        "source_partition": "validation",
        "dataset_fingerprint": fingerprint,
        "evaluated_ids": predictions["id"].tolist(),
        "decoding": "chat_template/greedy/max_new_tokens=8",
        "max_items": max_items,
        "inference_seconds": round(inference_seconds, 3),
        "elapsed_seconds": round(elapsed_seconds, 3),
        "peak_gpu_allocated_gib": round(peak_gpu_gib, 3) if peak_gpu_gib is not None else None,
    }
    (RUNS / "predictions_sft_validation_manifest.json").write_text(
        json.dumps(protocol, indent=2) + "\n", encoding="utf-8"
    )
    print("Inference seconds:", round(inference_seconds, 3), "Peak GPU allocated GiB:", protocol["peak_gpu_allocated_gib"])
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Model inference skipped; set RUN_MODEL_EVAL=1 after completed SFT training.")

Source mode: existing Hugging Face Arrow cache
train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


SFT examples completed: 20/121


...

SFT examples completed: 80/121


SFT examples completed: 100/121


SFT examples completed: 120/121
SFT examples completed: 121/121
Sample size: 121
Reference distribution: {'yes': 64, 'no': 40, 'maybe': 17}
Predicted distribution: {'yes': 81, 'no': 40}
Summary: {'n': 121, 'accuracy': 0.6033057851239669, 'macro_f1': 0.421551724137931, 'valid_format': 1.0, 'majority_label': 'yes', 'majority_accuracy': 0.5289256198347108, 'uniform_random_accuracy': 0.3333333333333333}
Confusion matrix (rows=reference, columns=prediction):
         yes  no  maybe  invalid
yes       50  14      0        0
no        17  23      0        0
maybe     14   3      0        0
invalid    0   0      0        0
Error samples (inspect generation and label handling):


,id,label,prediction,raw,input_token_count
0,10135926,yes,no,no,329
4,11458136,maybe,no,no,314
6,11713724,no,yes,yes,433
7,11926574,no,yes,yes,436
11,12419743,yes,no,no,421
13,12805495,maybe,yes,yes,383
15,12920330,maybe,yes,yes,283
21,16498158,yes,no,no,453


Inference seconds: 6.956 Peak GPU allocated GiB: 0.958


### Immediate interpretation of SFT inference

The saved SFT run answered 73 of 121 validation questions correctly, with macro-F1 0.422. Its valid-format rate was 1.0, yet it produced no maybe label for 17 reference-maybe cases. The evaluation is meaningful only against the matched base rows and majority reference checked below.

### Compare only evaluations with matching IDs and protocol manifests

In [7]:
from pathlib import Path

base_file = RUNS / "predictions_base_validation.csv"
sft_file = RUNS / "predictions_sft_validation.csv"

if base_file.exists() and sft_file.exists():
    base_meta = json.loads((RUNS / "predictions_base_validation_manifest.json").read_text())
    sft_meta = json.loads((RUNS / "predictions_sft_validation_manifest.json").read_text())
    for key in ("model_id", "model_revision", "dataset_fingerprint", "evaluated_ids", "decoding", "max_items"):
        if base_meta[key] != sft_meta[key]:
            raise ValueError(f"Cannot compare incompatible runs: {key} differs.")

    base = pd.read_csv(base_file)
    sft = pd.read_csv(sft_file)
    print("Starting-model metrics:", assess_predictions(base))
    print("SFT metrics:", assess_predictions(sft))
    aligned = base[["id", "label", "prediction"]].merge(
        sft[["id", "label", "prediction"]],
        on=["id", "label"], validate="one_to_one", suffixes=("_base", "_sft"),
    )
    assert len(aligned) == len(base) == len(sft)
    print("Difference in accuracy:",
          round((aligned["prediction_sft"].eq(aligned["label"]).mean()
                - aligned["prediction_base"].eq(aligned["label"]).mean()), 4))
else:
    print("Run notebooks 04 and 06 with matching settings before comparing models.")

Starting-model metrics: {'n': 121, 'accuracy': 0.15702479338842976, 'macro_f1': 0.11507936507936507, 'valid_format': 1.0, 'majority_label': 'yes', 'majority_accuracy': 0.5289256198347108, 'uniform_random_accuracy': 0.3333333333333333}
SFT metrics: {'n': 121, 'accuracy': 0.6033057851239669, 'macro_f1': 0.421551724137931, 'valid_format': 1.0, 'majority_label': 'yes', 'majority_accuracy': 0.5289256198347108, 'uniform_random_accuracy': 0.3333333333333333}
Difference in accuracy: 0.4463


### Results

SFT scored 73/121 (60.3%) on the same validation IDs versus 19/121 (15.7%) for base. Macro-F1 was 0.422 and format validity 100%. The majority-yes rule scored 64/121 (52.9%). SFT predicted yes 81 times, no 40, and maybe zero, missing all 17 true-maybe cases.

### Interpretation

The matched gain over base is substantial, but the nine-case advantage over the majority rule is uncertain (two-sided paired p about 0.188).

### Limitations

Repeatedly inspected validation data and zero maybe recall prevent a balanced three-class or deployment-quality claim.

### Key Observations

Report classwise behavior and both baselines whenever presenting the SFT accuracy.


## Key observations
The validation sample and settings must match exactly. Changes to tokenizer, model ID, prompt, maximum tokens, or selection of examples may produce a result that is **not** attributable to fine-tuning.

## Q&A
**Question:** Is a valid response format proof that SFT improved biomedical accuracy?  
**Answer:** No. Format validity and correctness measure different properties; both need separate inspection and a comparison to the majority-class baseline.